In [12]:
import pandas as pd
import sqlalchemy

engine_apt = sqlalchemy.create_engine("sqlite+pysqlite:///data/appatments.db")
engine_orders = sqlalchemy.create_engine("sqlite+pysqlite:///data/orders_by_time_and_customers.db")

inspector_apt = sqlalchemy.inspect(engine_apt)
print(inspector_apt.get_table_names())
inspector_apt.get_columns("appatments")

apt_df = pd.read_sql("SELECT * FROM appatments ORDER BY first_day_exposition", con=engine_apt)
orders_df = pd.read_sql("SELECT * FROM orders", con=engine_orders)

display(apt_df.head(10))
display(orders_df.head(10))

['appatments']


,last_price,total_area,first_day_exposition,rooms,ceiling_height,floors_total,living_area,floor,studio,kitchen_area,balcony,locality_name,days_exposition
0,1968750.0,48.20,2021-01-18 00:00:00.000000,2,2.50,5,27.4,2,0,7.7,3.0,Boryspil,1580.0
1,2475000.0,38.63,2021-05-26 00:00:00.000000,1,2.85,25,15.0,6,0,12.3,2.0,Vyshneve,1452.0
2,1912500.0,54.60,2021-07-04 00:00:00.000000,2,2.50,5,29.7,1,0,8.5,2.0,Boryspil,1413.0
3,16031250.0,270.00,2021-08-21 00:00:00.000000,16,3.00,4,180.0,4,0,13.0,1.0,Kyiv,1365.0
4,3768188.0,77.50,2021-08-25 00:00:00.000000,2,2.75,4,34.9,4,0,15.6,1.0,Borshchahivka,1361.0
5,7875000.0,142.00,2021-09-14 00:00:00.000000,3,2.77,5,95.7,4,0,14.3,1.0,Kyiv,1341.0
6,8718750.0,121.00,2021-09-23 00:00:00.000000,3,2.70,23,55.5,22,0,29.3,5.0,Kyiv,1332.0
7,1963125.0,43.00,2021-10-16 00:00:00.000000,2,2.50,5,28.0,2,0,5.0,1.0,Brovary,1309.0
8,18000000.0,235.00,2021-11-19 00:00:00.000000,5,3.75,5,160.0,3,0,23.0,4.0,Kyiv,1275.0
9,1563750.0,42.60,2022-01-04 00:00:00.000000,2,2.50,4,28.7,3,0,6.0,1.0,Boryspil,1229.0


,InvoiceNo,CustomerID,Description,Quantity,UnitPrice,Category,Discount,PaymentMethod,order_date
0,INV100290,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,USB-C Cable,8,65.59,Accessories,0.22,credit card,2024-11-06 00:00:00.000000
1,INV100278,0d88c95d-0f2f-4e4b-98c4-427c91125f7d,Gaming Keyboard,2,14.44,Gaming,0.07,paypal,2024-02-04 00:00:00.000000
2,INV100561,09c9ad28-738f-4e6c-b61b-6eda53404915,Gaming Keyboard,10,19.09,Gaming,0.24,paypal,2024-01-07 00:00:00.000000
3,INV100542,01eac1fb-42ac-4ad4-afd6-fe01d3e2dcf5,Power Bank,6,6.27,Accessories,0.06,credit card,2024-06-07 00:00:00.000000
4,INV100787,08508316-a3be-4bcd-b6b7-91145174a8ba,Power Bank,2,67.56,Accessories,0.15,credit card,2024-10-06 00:00:00.000000
5,INV100899,04e44f80-9f4e-4fc9-a45c-a81097d07a02,Laptop Stand,10,17.90,Office,0.25,paypal,2024-11-25 00:00:00.000000
6,INV100475,03020226-ebdd-49c3-a545-1d2c01a91620,Laptop Stand,7,8.11,Office,0.10,paypal,2024-04-10 00:00:00.000000
7,INV100724,10bf93a6-29de-4f41-afd0-e62610ac2068,Bluetooth Speaker,6,10.95,Electronics,0.16,bank transfer,2024-08-15 00:00:00.000000
8,INV100144,0a0ad13a-2fac-484f-b54c-9607340f0524,Wireless Mouse,7,54.45,Electronics,0.05,bank transfer,2024-09-18 00:00:00.000000
9,INV100766,045e6cd9-0bd9-4b88-808b-10c5a4a13d0d,Gaming Keyboard,1,56.38,Gaming,0.25,paypal,2024-01-21 00:00:00.000000


In [6]:
query = sqlalchemy.text("""
SELECT locality_name, COUNT(*) AS sold_count
FROM appatments
WHERE rooms = 2
AND strftime('%Y', first_day_exposition) = '2023'
GROUP BY locality_name
ORDER BY sold_count DESC
""")
task1 = pd.read_sql(query, con=engine_apt)
task1

,locality_name,sold_count
0,Kyiv,414
1,Boryspil,36
2,Vyshneve,34
3,Hostomel,34
4,Irpin,33
5,Brovary,28
6,Boyarka,28
7,Borshchahivka,28
8,Bucha,19


In [14]:
query = sqlalchemy.text("""
SELECT strftime('%Y', first_day_exposition) AS year, COUNT(*) AS sold_count
FROM appatments
WHERE rooms = 3 AND locality_name = 'Kyiv'
GROUP BY year
ORDER BY year
""")
task2 = pd.read_sql(query, con=engine_apt)
task2

,year,sold_count
0,2021,2
1,2022,23
2,2023,348
3,2024,681


In [15]:
query = sqlalchemy.text("""
SELECT locality_name, first_day_exposition
FROM appatments
WHERE first_day_exposition = (SELECT MAX(first_day_exposition) FROM appatments)
""")
task3 = pd.read_sql(query, con=engine_apt)
task3

,locality_name,first_day_exposition
0,Boryspil,2024-12-23 00:00:00.000000
1,Kyiv,2024-12-23 00:00:00.000000


In [16]:
query = sqlalchemy.text("""
SELECT c.Location, COUNT(*) AS orders_count
FROM orders o
JOIN customers c ON o.CustomerID = c.CustomerID
WHERE strftime('%Y-%m', o.order_date) = '2024-03'
GROUP BY c.Location
ORDER BY orders_count DESC
""")
task4 = pd.read_sql(query, con=engine_orders)
task4

,Location,orders_count
0,Kyiv,4
1,Brovary,1
2,Boryspil,1


In [19]:
import plotly.express as px

query = sqlalchemy.text("""
SELECT strftime('%Y-%m', o.order_date) AS month, COUNT(*) AS orders_count
FROM orders o
JOIN customers c ON o.CustomerID = c.CustomerID
WHERE c.Location = 'Kyiv'
GROUP BY month
ORDER BY month
""")
task5 = pd.read_sql(query, con=engine_orders)

fig = px.line(task5, x='month', y='orders_count', markers=True,
              title='Количество заказов по месяцам — Kyiv, 2024',
              labels={'month': 'Месяц', 'orders_count': 'Заказов'})
fig.show()

In [21]:
query = sqlalchemy.text("""
SELECT c.Location, SUM(o.Quantity) AS total_qty
FROM orders o
JOIN customers c ON o.CustomerID = c.CustomerID
WHERE o.Description = 'Gaming Keyboard' AND strftime('%Y-%m', o.order_date) = '2024-08'
GROUP BY c.Location
ORDER BY total_qty DESC
""")
task6 = pd.read_sql(query, con=engine_orders)
task6

,Location,total_qty
